In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/nowe/gryc_data.csv'
df = pd.read_csv(file_path)

print(df.head())

   Store_ID                     Store_Name Zone_Time Zone_Type  Area_km2  \
0         1  Grycan - Rynek (Stare Miasto)   0-5 min      Core      0.38   
1         1  Grycan - Rynek (Stare Miasto)  5-10 min      Ring      1.12   
2         2          Grycan - CH Wroclavia   0-5 min      Core      0.44   
3         2          Grycan - CH Wroclavia  5-10 min      Ring      1.26   
4         3  Grycan - CH Pasaz Grunwaldzki   0-5 min      Core      0.41   

   Population  Competitor_Count  RCD_Density_per_km2  SCR_Overlap_Pct  \
0        4210                14                36.84             88.2   
1       11850                26                23.21             79.4   
2        5120                 9                20.45             64.5   
3       14300                31                24.60             82.1   
4        4680                 8                19.51             61.0   

   CPV_Capita_Per_Venue  CER_Erosion_Pct   Market_Classification  
0                   281             8

In [3]:
# ==============================================================================
# 0. INSTALACJA I IMPORT PAKIETÓW GEOPRZESTRZENNYCH
# ==============================================================================
!pip install geopandas shapely pyproj -q

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon

In [4]:
# ==============================================================================
# 1. PARAMETRY I WCZYTANIE PLIKÓW SHAPEFILE
# ==============================================================================
# Docelowy układ metryczny: ETRF2000-PL / CS2000 zone 6 (EPSG:2177 dla Wrocławia)
TARGET_CRS = "EPSG:2177"

# AKTUALIZACJA ŚCIEŻKI DO PLIKÓW SHAPEFILE DYSKU GOOGLE
PATH_GRYCAN = "/content/drive/MyDrive/Colab Notebooks/nowe/grycan.shp"
PATH_KONKURENCJA = "/content/drive/MyDrive/Colab Notebooks/nowe/konkurencyja.shp"
PATH_REJURB = "/content/drive/MyDrive/Colab Notebooks/nowe/REJURB_20251231.shp"

print("Wczytywanie warstw wektorowych...")
gdf_grycan = gpd.read_file(PATH_GRYCAN)
gdf_konkurencja = gpd.read_file(PATH_KONKURENCJA)
gdf_rejurb = gpd.read_file(PATH_REJURB)

# Projekcja do metrycznego układu płaskiego
gdf_grycan = gdf_grycan.to_crs(TARGET_CRS)
gdf_konkurencja = gdf_konkurencja.to_crs(TARGET_CRS)
gdf_rejurb = gdf_rejurb.to_crs(TARGET_CRS)

# Automatyczna naprawa ewentualnych samoprzecięć w geometriach
gdf_grycan['geometry'] = gdf_grycan['geometry'].buffer(0)
gdf_konkurencja['geometry'] = gdf_konkurencja['geometry'].buffer(0)
gdf_rejurb['geometry'] = gdf_rejurb['geometry'].buffer(0)

Wczytywanie warstw wektorowych...


In [5]:


# ==============================================================================
# 1. PARAMETRY I WCZYTANIE PLIKÓW SHAPEFILE
# ==============================================================================
# Docelowy układ metryczny: ETRF2000-PL / CS2000 zone 6 (EPSG:2177 dla Wrocławia)
TARGET_CRS = "EPSG:2177" # Dodałem brakującą definicję TARGET_CRS

# AKTUAKLIZACJA ŚCIEŻKI DO PLIKÓW SHAPEFILE DYSKU GOOGLE
PATH_GRYCAN = "/content/drive/MyDrive/Colab Notebooks/nowe/grycan.shp"
PATH_KONKURENCJA = "/content/drive/MyDrive/Colab Notebooks/nowe/konkurencyja.shp"
PATH_REJURB = "/content/drive/MyDrive/Colab Notebooks/nowe/REJURB_20251231.shp"

print("Wczytywanie warstw wektorowych...")
gdf_grycan = gpd.read_file(PATH_GRYCAN)
gdf_konkurencja = gpd.read_file(PATH_KONKURENCJA)
gdf_rejurb = gpd.read_file(PATH_REJURB)

# Projekcja do metrycznego układu płaskiego
gdf_grycan = gdf_grycan.to_crs(TARGET_CRS)
gdf_konkurencja = gdf_konkurencja.to_crs(TARGET_CRS)
gdf_rejurb = gdf_rejurb.to_crs(TARGET_CRS)

# Automatyczna naprawa ewentualnych samoprzecięć w geometriach
gdf_grycan['geometry'] = gdf_grycan['geometry'].buffer(0)
gdf_konkurencja['geometry'] = gdf_konkurencja['geometry'].buffer(0)
gdf_rejurb['geometry'] = gdf_rejurb['geometry'].buffer(0)

# ==============================================================================
# 2. IDENTYFIKACJA KOLUMN ATRYBUTOWYCH
# ==============================================================================
# Dynamiczne mapowanie kolumny z gęstością zaludnienia w REJURB
density_col_candidates = [c for c in gdf_rejurb.columns if any(k in c.lower() for k in ['gest', 'dens', 'lud', 'pop', 'osob'])]
DENSITY_COL = density_col_candidates[0] if density_col_candidates else gdf_rejurb.select_dtypes(include=[np.number]).columns[0]
print(f"-> Zidentyfikowano pole gęstości zaludnienia w REJURB: '{DENSITY_COL}'")

# Dynamiczne mapowanie identyfikatora/nazwy lokalu Grycana
id_col_candidates = [c for c in gdf_grycan.columns if any(k in c.lower() for k in ['nazw', 'name', 'id', 'lokal', 'store'])]
STORE_ID_COL = id_col_candidates[0] if id_col_candidates else gdf_grycan.columns[0]

# Identyfikacja kolumny czasu dojścia (0-5 vs 5-10 / 5 vs 10 min)
time_col_candidates = [c for c in gdf_grycan.columns if any(k in c.lower() for k in ['czas', 'time', 'min', 'strefa', 'zone', 'range'])]
TIME_COL = time_col_candidates[0] if time_col_candidates else None

# ==============================================================================
# 3. WYODRĘBNIENIE STREF: RDZEŃ (0-5 min) ORAZ PIERŚCIEŃ (5-10 min)
# ==============================================================================
records = []

# Łączna geometria kawiarni konkurencji do badania nakładania przestrzennego (SCR)
union_konkurencja = gdf_konkurencja.geometry.union_all()

# Punkty reprezentatywne (centra kawiarni) do precyzyjnego zliczania lokali w pierścieniach
pts_konkurencja = gdf_konkurencja.copy()
pts_konkurencja['geom_pt'] = pts_konkurencja.geometry.centroid
pts_konkurencja = pts_konkurencja.set_geometry('geom_pt')

unique_stores = gdf_grycan[STORE_ID_COL].unique()
print(f"Przetwarzanie {len(unique_stores)} lokalizacji Grycana...")

for store in unique_stores:
    store_geoms = gdf_grycan[gdf_grycan[STORE_ID_COL] == store].copy()

    # Sortowanie/rozdzielenie stref wg czasu dojścia
    if TIME_COL:
        geom_5 = store_geoms[store_geoms[TIME_COL].astype(str).str.contains('5') & ~store_geoms[TIME_COL].astype(str).str.contains('10')].geometry.unary_union
        geom_10 = store_geoms[store_geoms[TIME_COL].astype(str).str.contains('10')].geometry.unary_union
    else:
        # Fallback: mniejszy poligon = 5 min, większy = 10 min

        store_geoms['area'] = store_geoms.geometry.area
        sorted_by_area = store_geoms.sort_values(by='area')
        geom_5 = sorted_by_area.iloc[0].geometry
        geom_10 = sorted_by_area.iloc[-1].geometry if len(sorted_by_area) > 1 else geom_5

    # Konstrukcja pierścienia zewnętrznego (5-10 min) bez środka
    ring_5_10 = geom_10.difference(geom_5) if (geom_10 and not geom_10.is_empty and geom_10 != geom_5) else Polygon()

    # Pętla po obu pasmach
    zones_to_process = [
        ('0-5 min', 'Core', geom_5),
        ('5-10 min', 'Ring', ring_5_10)
    ]

    for zone_label, zone_type, geom in zones_to_process:
        if geom is None or geom.is_empty:
            continue

        area_m2 = geom.area
        area_km2 = area_m2 / 1_000_000.0

        # A. Interpolacja powierzchniowa ludności (Areal Interpolation)
        sub_gdf = gpd.GeoDataFrame([{'geometry': geom}], crs=TARGET_CRS)
        intersect_rej = gpd.overlay(sub_gdf, gdf_rejurb, how='intersection')

        if not intersect_rej.empty:
            intersect_rej['part_area_km2'] = intersect_rej.geometry.area / 1_000_000.0
            # Mnożenie powierzchni wycinka przez przypisaną gęstość rejonu
            est_population = int(np.round((intersect_rej['part_area_km2'] * intersect_rej[DENSITY_COL]).sum()))
        else:
            est_population = 0

        #  B. Analiza presji konkurencyjnej
        # Liczba punktów konkurencji wewnątrz strefy
        comp_in_zone = pts_konkurencja[pts_konkurencja.geometry.within(geom)]
        comp_count = len(comp_in_zone)

        # Zagęszczenie konkurencji na km²
        rcd_density = comp_count / area_km2 if area_km2 > 0 else 0.0

        # Powierzchnia i procent nakładania się izochron (Spatial Contestation Ratio)
        overlap_geom = geom.intersection(union_konkurencja)
        overlap_area_km2 = overlap_geom.area / 1_000_000.0 if overlap_geom else 0.0
        scr_overlap_pct = (overlap_area_km2 / area_km2 * 100.0) if area_km2 > 0 else 0.0

        # Chłonność demograficzna na punkt (Capita Per Venue)
        cpv_capita = int(np.round(est_population / (1 + comp_count)))

        # Erozja bazy popytowej (szacunek populacji w strefie spornej)
        if not overlap_geom.is_empty:
            sub_overlap = gpd.GeoDataFrame([{'geometry': overlap_geom}], crs=TARGET_CRS)
            intersect_overlap = gpd.overlay(sub_overlap, gdf_rejurb, how='intersection')
            if not intersect_overlap.empty:
                intersect_overlap['area_km2'] = intersect_overlap.geometry.area / 1_000_000.0
                overlap_pop = (intersect_overlap['area_km2'] * intersect_overlap[DENSITY_COL]).sum()
                cer_erosion_pct = min(100.0, (overlap_pop / est_population * 100.0)) if est_population > 0 else 0.0
            else:
                cer_erosion_pct = 0.0
        else:
            cer_erosion_pct = 0.0

        records.append({
            'Store_ID': store,
            'Store_Name': f"Grycan - {store}",
            'Zone_Time': zone_label,
            'Zone_Type': zone_type,
            'Area_km2': np.round(area_km2, 2),
            'Population': est_population,
            'Competitor_Count': comp_count,
            'RCD_Density_per_km2': np.round(rcd_density, 2),
            'SCR_Overlap_Pct': np.round(scr_overlap_pct, 1),
            'CPV_Capita_Per_Venue': cpv_capita,
            'CER_Erosion_Pct': np.round(cer_erosion_pct, 1)
        })

df_results = pd.DataFrame(records)

# ==============================================================================
# 4. KLASYFIKACJA GEOMARKETINGOWA LOKALI (TYPOLOGIA RELACJI PRZESTRZENNYCH)
# ==============================================================================
def classify_market(row):
    store = row['Store_ID']
    store_data = df_results[df_results['Store_ID'] == store]
    rcd_5 = store_data[store_data['Zone_Time'] == '0-5 min']['RCD_Density_per_km2'].values
    rcd_10 = store_data[store_data['Zone_Time'] == '5-10 min']['RCD_Density_per_km2'].values

    val_5 = rcd_5[0] if len(rcd_5) > 0 else 0.0
    val_10 = rcd_10[0] if len(rcd_10) > 0 else 0.0

    if val_5 >= 20.0 and val_10 >= 20.0:
        return 'Klastrowanie agresywne (Rdzen miejski)'
    elif val_5 < 15.0 and val_10 >= 15.0:
        return 'Bariera pierscieniowa'
    elif val_5 < 5.0 and val_10 < 5.0:
        return 'Monopol lokalny'
    else:
        return 'Enklawa rynkowa'

df_results['Market_Classification'] = df_results.apply(classify_market, axis=1)

# ==============================================================================
# 5. EKSFORT DANYCH TABELARYCZNYCH I PRZESTRZENNYCH DLA TABLEAU
# ==============================================================================
CSV_OUTPUT = "Grycan_Spatial_BI_Calculated.csv"
GEOJSON_OUTPUT = "Grycan_Zones_Calculated.geojson"

df_results.to_csv(CSV_OUTPUT, index=False, encoding='utf-8')
print(f"\n[SUKCES] Zapisano przeliczone wskaźniki do: {CSV_OUTPUT}")

# Eksport geometrii stref (GeoJSON) z dołączonymi wskaźnikami pod bezpośrednie mapowanie w Tableau
gdf_export_list = []
for r in records:
    # Pobranie odpowiedniej geometrii
    st = r['Store_ID']
    zt = r['Zone_Time']
    st_geoms = gdf_grycan[gdf_grycan[STORE_ID_COL] == st].copy()

    # Dodanie tymczasowej kolumny z powierzchnią, aby móc ją sortować
    st_geoms['area'] = st_geoms.geometry.area

    if zt == '0-5 min':
        g = st_geoms.sort_values(by='area').iloc[0].geometry
    else:
        g1 = st_geoms.sort_values(by='area').iloc[0].geometry
        g2 = st_geoms.sort_values(by='area').iloc[-1].geometry
        g = g2.difference(g1) if g2 != g1 else g1

    r_dict = dict(r)
    r_dict['geometry'] = g
    gdf_export_list.append(r_dict)

gdf_tableau_map = gpd.GeoDataFrame(gdf_export_list, crs=TARGET_CRS).to_crs("EPSG:4326")
gdf_tableau_map.to_file(GEOJSON_OUTPUT, driver='GeoJSON')
print(f"[SUKCES] Zapisano warstwę wektorową stref do: {GEOJSON_OUTPUT}")

# Podgląd wynikowej tabeli
print("\n--- PODGLĄD WYNIKÓW ---")
print(df_results[['Store_ID', 'Zone_Time', 'Population', 'Competitor_Count', 'RCD_Density_per_km2', 'CPV_Capita_Per_Venue', 'Market_Classification']])

Wczytywanie warstw wektorowych...
-> Zidentyfikowano pole gęstości zaludnienia w REJURB: 'LUD_NA_KM2'
Przetwarzanie 1 lokalizacji Grycana...

[SUKCES] Zapisano przeliczone wskaźniki do: Grycan_Spatial_BI_Calculated.csv
[SUKCES] Zapisano warstwę wektorową stref do: Grycan_Zones_Calculated.geojson

--- PODGLĄD WYNIKÓW ---
   Store_ID Zone_Time  Population  Competitor_Count  RCD_Density_per_km2  \
0         0   0-5 min        8228                 0                  0.0   
1         0  5-10 min       26921                 0                  0.0   

   CPV_Capita_Per_Venue Market_Classification  
0                  8228       Monopol lokalny  
1                 26921       Monopol lokalny  


In [6]:
unique_store_ids_grycan_shp = gdf_grycan[STORE_ID_COL].unique()
num_unique_stores_grycan_shp = len(unique_store_ids_grycan_shp)

print(f"Liczba unikalnych Store_ID (FacilityID) w grycan.shp: {num_unique_stores_grycan_shp}")
print(f"Unikalne Store_ID (FacilityID) w grycan.shp: {unique_store_ids_grycan_shp}")

Liczba unikalnych Store_ID (FacilityID) w grycan.shp: 1
Unikalne Store_ID (FacilityID) w grycan.shp: [0]


In [7]:
# Wczytanie obu warstw
gdf_grycan = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/nowe/grycan.shp")
gdf_konkurencja = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/nowe/konkurencyja.shp")

# Ujednolicenie układu (np. WGS84 - EPSG:4326 pod Tableau)
gdf_grycan = gdf_grycan.to_crs("EPSG:4326")
gdf_konkurencja = gdf_konkurencja.to_crs("EPSG:4326")

# Dodanie kolumny kategoryzującej
gdf_grycan['Siec'] = 'Grycan'
gdf_konkurencja['Siec'] = 'Konkurencja'

# Nadanie unikalnych identyfikatorów
gdf_grycan['Lokal_ID'] = [f"G_{i+1}" for i in range(len(gdf_grycan))]
gdf_konkurencja['Lokal_ID'] = [f"K_{i+1}" for i in range(len(gdf_konkurencja))]

# Scalenie pionowe (Union)
gdf_all = pd.concat([
    gdf_grycan[['Lokal_ID', 'Siec', 'geometry']],
    gdf_konkurencja[['Lokal_ID', 'Siec', 'geometry']]
], ignore_index=True)

gdf_all = gpd.GeoDataFrame(gdf_all, crs="EPSG:4326")
gdf_all.to_file("Kawiarnie_Wroclaw_Wszystkie.geojson", driver="GeoJSON")
print("Gotowe! Pobierz plik 'Kawiarnie_Wroclaw_Wszystkie.geojson'.")

Gotowe! Pobierz plik 'Kawiarnie_Wroclaw_Wszystkie.geojson'.


In [8]:
from google.colab import files

files.download('Kawiarnie_Wroclaw_Wszystkie.geojson')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Geokodowanie lodziarni

In [10]:
# ==============================================================================
# PROJEKT GEOMARKETINGOWY: GEOKODOWANIE LODZIARNI (WROCŁAW + BIELANY WROCŁAWSKIE)
# Moduł: Pozyskanie danych POI (Overpass API / OpenStreetMap) do analizy konkurencji
# ==============================================================================

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import time

# KROK 2: Przygotowanie zapytania Overpass QL
# Bounding box obejmujący Wrocław oraz południowy węzeł handlowy w Bielanach Wrocławskich
BBOX = "51.0200,16.8000,51.2200,17.1800"

overpass_query = f"""
[out:json][timeout:60];
(
  node["amenity"="ice_cream"]({BBOX});
  way["amenity"="ice_cream"]({BBOX});
  node["cuisine"="ice_cream"]({BBOX});
  way["cuisine"="ice_cream"]({BBOX});
  node["shop"="ice_cream"]({BBOX});
  way["shop"="ice_cream"]({BBOX});
);
out center tags;
"""

# Lista publicznych instancji Overpass API dla zachowania ciągłości działania w przypadku błędu 502
overpass_urls = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter"
]

headers = {'User-Agent': 'Colab Notebook Geomarketing/1.0'}
data = None

for url in overpass_urls:
    try:
        print(f"Wysyłanie zapytania do Overpass API ({url})...")
        response = requests.post(url, data={'data': overpass_query}, headers=headers, timeout=60)
        if response.status_code == 200:
            data = response.json()
            break
        else:
            print(f"Serwer {url} zwrócił status: {response.status_code}. Próba kolejnego serwera...")
    except Exception as e:
        print(f"Błąd połączenia z {url}: {e}. Próba kolejnego serwera...")
    time.sleep(1)

if data is None:
    raise ConnectionError("Wszystkie dostępne serwery Overpass API zwróciły błąd lub są niedostępne. Spróbuj ponownie później.")

elements = data.get('elements', [])
print(f"Pobrano {len(elements)} surowych obiektów z bazy OpenStreetMap.")

# KROK 3: Parsowanie i ustrukturyzowanie atrybutów
records = []
for el in elements:
    osm_id = el.get('id')
    tags = el.get('tags', {})

    # Wyciągnięcie współrzędnych (dla węzłów z lat/lon, dla poligonów z wyliczonego 'center')
    if el.get('type') == 'node':
        lat = el.get('lat')
        lon = el.get('lon')
    elif 'center' in el:
        lat = el['center'].get('lat')
        lon = el['center'].get('lon')
    else:
        continue

    name = tags.get('name', 'Lodziarnia bez nazwy')
    brand = tags.get('brand', tags.get('operator', 'Niezależna'))
    street = tags.get('addr:street', '')
    housenumber = tags.get('addr:housenumber', '')
    city = tags.get('addr:city', 'Wrocław/Kobierzyce')

    records.append({
        'osm_id': osm_id,
        'name': name,
        'brand': brand,
        'street': f"{street} {housenumber}".strip(),
        'city': city,
        'competitor_type': 'Ice_Cream',
        'lat': lat,
        'lon': lon
    })

# Tworzenie ramki danych Pandas
df = pd.DataFrame(records)

# Usunięcie lodziarni bez nazwy (te, które miały brakujący tag 'name' w OSM)
df = df[df['name'] != 'Lodziarnia bez nazwy'].reset_index(drop=True)

# Usunięcie lodziarni 'Grycan' z danych
df = df[~df['name'].str.contains('Grycan', case=False, na=False)].reset_index(drop=True)

# Czyszczenie duplikatów
df = df.drop_duplicates(subset=['osm_id']).reset_index(drop=True)
print(f"Liczba unikalnych punktów po usunięciu duplikatów: {len(df)}")

# KROK 4: Konwersja do warstwy przestrzennej GeoPandas i reprojekcja do PL-2000 strefa 6 (EPSG:2177)
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Transformacja współrzędnych metrycznych (EPSG:2177)
gdf_2177 = gdf.to_crs(epsg=2177)
gdf['x_2177'] = gdf_2177.geometry.x.round(2)
gdf['y_2177'] = gdf_2177.geometry.y.round(2)

# KROK 5: Zapis do plików
gdf.to_csv('lodziarnie_wroclaw_bielany.csv', index=False, encoding='utf-8-sig')
gdf.to_file('lodziarnie_wroclaw_bielany.geojson', driver='GeoJSON')
gdf_2177.to_file('lodziarnie_2177.shp', encoding='utf-8')

print("\n--- Zakończono sukcesem! ---")
print("Utworzono pliki:")
print("1. lodziarnie_wroclaw_bielany.csv (z koordynatami WGS84 oraz EPSG:2177)")
print("2. lodziarnie_wroclaw_bielany.geojson")
print("3. lodziarnie_2177.shp (warstwa Shapefile w układzie PL-2000)")

# Podgląd pierwszych 5 rekordów
gdf[['osm_id', 'name', 'brand', 'street', 'lat', 'lon', 'x_2177', 'y_2177']].head()

Wysyłanie zapytania do Overpass API (https://overpass-api.de/api/interpreter)...
Pobrano 98 surowych obiektów z bazy OpenStreetMap.
Liczba unikalnych punktów po usunięciu duplikatów: 85

--- Zakończono sukcesem! ---
Utworzono pliki:
1. lodziarnie_wroclaw_bielany.csv (z koordynatami WGS84 oraz EPSG:2177)
2. lodziarnie_wroclaw_bielany.geojson
3. lodziarnie_2177.shp (warstwa Shapefile w układzie PL-2000)


/tmp/ipykernel_557/1370359723.py:117: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_2177.to_file('lodziarnie_2177.shp', encoding='utf-8')
/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'competitor_type' to 'competitor'
  ogr_write(


,osm_id,name,brand,street,lat,lon,x_2177,y_2177
0,1282020956,Polish Lody,Niezależna,,51.108518,17.034815,6432410.29,5664165.28
1,1368344721,Mocna Cafe,Niezależna,Edwarda Dembowskiego 55,51.106125,17.103676,6437229.18,5663838.12
2,1496242302,beza krówka,Niezależna,,51.087638,17.019380,6431298.48,5661856.90
3,2406712225,Brama Michalaka,Niezależna,,51.145992,17.127761,6438968.38,5668252.66
4,2454663885,Balduno,Niezależna,,51.116352,17.041963,6432922.22,5665030.21


In [ ]:
from google.colab import files
import zipfile
import os

# Lista plików wchodzących w skład pakietu Shapefile
shp_files = [
    'lodziarnie_2177.shp',
    'lodziarnie_2177.shx',
    'lodziarnie_2177.dbf',
    'lodziarnie_2177.prj',
    'lodziarnie_2177.cpg'
]

zip_filename = 'lodziarnie_2177_shp_package.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in shp_files:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))
        else:
            print(f"Warning: File not found: {file}. It will not be included in the zip archive.")

files.download(zip_filename)
